# Within-Corpus Spam Classification — Organized Notebook

This notebook runs **one feature configuration at a time**. Select the configuration in the settings cell below, then restart the kernel and run all cells whenever you change it.

The evaluation setup is kept consistent with the existing notebook: five classifiers, the existing hyperparameters, `RepeatedStratifiedKFold` with 10 folds × 3 repetitions, and accuracy / precision / recall / F1 reporting.

**Input expectation:** the input pickle is the experiment-ready 30,000-row Dataset B sample and contains the preprocessed `text` column, the six metadata columns, and the binary `label` column. This notebook does not rebuild those features from raw email messages.

## 1. Imports

In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

## 2. Settings — choose one configuration

In [ ]:
# Choose exactly one: "Text", "Metadata", or "Text + Metadata"
CONFIGURATION = "Text"

# Dataset B sample path seen in the original notebook's Kaggle environment.
# Change this path if your input pickle is stored elsewhere.
DATA_PATH = "/kaggle/input/datasets/kmanolak/sample2/spam_sample_30000_dataset2.pkl"

# Shared experiment settings
RANDOM_STATE = 42
N_SPLITS = 10
N_REPEATS = 3
TFIDF_MAX_FEATURES = 20000
TFIDF_NGRAM_RANGE = (1, 2)

VALID_CONFIGURATIONS = {"Text", "Metadata", "Text + Metadata"}
if CONFIGURATION not in VALID_CONFIGURATIONS:
    raise ValueError(
        f"CONFIGURATION must be one of {sorted(VALID_CONFIGURATIONS)}; got {CONFIGURATION!r}."
    )

metadata_features = [
    "num_exclamations",
    "num_dollars",
    "num_urls",
    "num_emails",
    "num_hashes",
    "num_digits",
]

print("Selected configuration:", CONFIGURATION)
print("Input path:", DATA_PATH)

## 3. Load and validate the experiment-ready dataset

The original notebook’s data-loading cell was blank (`pd.read_pickle()` without a path). The path below is filled using the Dataset B sample path recorded in that notebook. This cell deliberately does not impute or modify missing values; it reports them so the actual preprocessing decision is not guessed.

In [ ]:
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Could not find the input pickle at {DATA_PATH!r}. "
        "Update DATA_PATH in the settings cell to the location of your 30,000-row Dataset B sample."
    )

df = pd.read_pickle(DATA_PATH).copy()

# Accommodate the alternative text-column name used in an earlier notebook cell.
if "text" not in df.columns and "text_combined" in df.columns:
    df = df.rename(columns={"text_combined": "text"})

required_columns = ["text", "label", *metadata_features]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(
        "The loaded dataframe is missing required columns: "
        f"{missing_columns}. Available columns: {df.columns.tolist()}"
    )

print("Dataset shape:", df.shape)
print("Missing values in required columns:")
print(df[required_columns].isna().sum())
print("\nLabel counts:")
print(df["label"].value_counts(dropna=False))

if df[required_columns].isna().any().any():
    raise ValueError(
        "Missing values were found in required columns. Resolve them using the actual documented "
        "preprocessing procedure before running the evaluation; this notebook does not assume an imputation method."
    )

y_sample = df["label"].astype(int)
if not set(y_sample.unique()).issubset({0, 1}):
    raise ValueError(f"Expected binary labels 0/1, found: {sorted(y_sample.unique())}")

print("\nClass proportions:")
print(y_sample.value_counts(normalize=True))

## 4. Prepare the selected feature representation

- **Text:** `df["text"]` with TF-IDF.
- **Metadata:** the six metadata columns, passed through unchanged (`"passthrough"`). No `StandardScaler` is used.
- **Text + Metadata:** `df.drop(columns=["label"])`, with TF-IDF applied to `text` and metadata passed through unchanged.

In [ ]:
if CONFIGURATION == "Text":
    X_sample = df["text"]
    preprocessor = TfidfVectorizer(
        max_features=TFIDF_MAX_FEATURES,
        ngram_range=TFIDF_NGRAM_RANGE,
    )

elif CONFIGURATION == "Metadata":
    X_sample = df[metadata_features]
    preprocessor = "passthrough"

else:  # CONFIGURATION == "Text + Metadata"
    X_sample = df.drop(columns=["label"]).copy()
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "text",
                TfidfVectorizer(
                    max_features=TFIDF_MAX_FEATURES,
                    ngram_range=TFIDF_NGRAM_RANGE,
                ),
                "text",
            ),
            ("meta", "passthrough", metadata_features),
        ]
    )

print("X type:", type(X_sample))
print("X shape:", X_sample.shape)
print("Labels:", y_sample.shape)
print("Preprocessor:", preprocessor)

## 5. Cross-validation setup

In [ ]:
cv = RepeatedStratifiedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE,
)

print(f"CV: {N_SPLITS} folds × {N_REPEATS} repetitions = {N_SPLITS * N_REPEATS} evaluation folds")

## 6. Classifiers and hyperparameters

The values below follow the final model dictionary in the uploaded notebook. They are not tuned or changed in this reorganization.

In [ ]:
models = {
    "Naive Bayes": MultinomialNB(alpha=1.0),

    "Linear SVM": LinearSVC(
        C=1.0,
        max_iter=5000,
        random_state=42,
    ),

    "Logistic Regression": LogisticRegression(
        solver="saga",
        C=0.1,
        max_iter=20000,
        tol=1e-3,
        random_state=42,
        n_jobs=-1,
    ),

    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        eval_metric="logloss",
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=1,
    ),
}

print("Models to evaluate:")
for model_name in models:
    print("-", model_name)

## 7. Run the selected configuration

Only the configuration selected in Section 2 is used. Each model is saved to the results list as it completes, and both the summary metrics and all-fold metrics are exported using configuration-specific filenames.

In [ ]:
scoring = ["accuracy", "precision", "recall", "f1"]
results = []
fold_results = {}

configuration_tag = CONFIGURATION.replace(" + ", "_Plus_").replace(" ", "_")
summary_path = f"DatasetB_{configuration_tag}_SummaryMetrics.csv"
folds_path = f"DatasetB_{configuration_tag}_AllFoldMetrics.csv"

for name, model in models.items():
    print(f"\nRunning {name} for configuration: {CONFIGURATION}...")

    pipeline = Pipeline([
        ("features", preprocessor),
        ("classifier", model),
    ])

    scores = cross_validate(
        pipeline,
        X_sample,
        y_sample,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
    )

    fold_results[f"{name}_Accuracy"] = scores["test_accuracy"]
    fold_results[f"{name}_Precision"] = scores["test_precision"]
    fold_results[f"{name}_Recall"] = scores["test_recall"]
    fold_results[f"{name}_F1"] = scores["test_f1"]

    result = {
        "Configuration": CONFIGURATION,
        "Model": name,
        "Accuracy Mean": np.mean(scores["test_accuracy"]),
        "Accuracy Std": np.std(scores["test_accuracy"]),
        "Precision Mean": np.mean(scores["test_precision"]),
        "Precision Std": np.std(scores["test_precision"]),
        "Recall Mean": np.mean(scores["test_recall"]),
        "Recall Std": np.std(scores["test_recall"]),
        "F1 Mean": np.mean(scores["test_f1"]),
        "F1 Std": np.std(scores["test_f1"]),
    }

    results.append(result)
    pd.DataFrame(results).to_csv(summary_path, index=False)
    pd.DataFrame(fold_results).to_csv(folds_path, index=False)

    print(result)
    print(f"Updated: {summary_path}")
    print(f"Updated: {folds_path}")

## 8. Review the results

In [ ]:
results_df = pd.DataFrame(results)
results_df

## Notes

1. To run a different setup, change `CONFIGURATION` to `"Text"`, `"Metadata"`, or `"Text + Metadata"`, then **restart the kernel and run all cells** so previous variables and outputs cannot interfere.
2. The notebook uses `"passthrough"` for metadata. It does not use `StandardScaler`.
3. The original notebook contained a blank `pd.read_pickle()` call and cells that redefined `preprocessor`, `models`, and `sample_df`. These are consolidated here to avoid accidental overwriting.
4. Before using the new exports in the manuscript, run the notebook and confirm the results match the intended experiment. Because the original notebook had overwritten variables and stale cell outputs, its prior executed state cannot be guaranteed to match a clean Run All.